In [ ]:
!pip install \
    rasterio \
    rioxarray \
    pyproj \
    --quiet --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 150.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 134.9 MB/s eta 0:00:00


In [ ]:
!wget https://challenge.ey.com/api/v1/storage/admin-files/556750475068843-678f9918ed9624f63ca75b56-Dataset.zip
!unzip 556750475068843-678f9918ed9624f63ca75b56-Dataset.zip
!rm 556750475068843-678f9918ed9624f63ca75b56-Dataset.zip
!mv "/content/Dataset/Training_data_uhi_index.csv" Training_data_uhi_index.csv
!mv "/content/Dataset/Submission_template.csv" Submission_template.csv

--2025-03-22 10:41:49--  https://challenge.ey.com/api/v1/storage/admin-files/556750475068843-678f9918ed9624f63ca75b56-Dataset.zip
Resolving challenge.ey.com (challenge.ey.com)... 52.236.158.32
Connecting to challenge.ey.com (challenge.ey.com)|52.236.158.32|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13861358 (13M) [application/octet-stream]
Saving to: ‘556750475068843-678f9918ed9624f63ca75b56-Dataset.zip’

556750475068843-678 100%[===================>]  13.22M  9.33MB/s    in 1.4s    

2025-03-22 10:41:51 (9.33 MB/s) - ‘556750475068843-678f9918ed9624f63ca75b56-Dataset.zip’ saved [13861358/13861358]

Archive:  556750475068843-678f9918ed9624f63ca75b56-Dataset.zip
   creating: Dataset/
  inflating: Dataset/2025 EY Open Science AI Data Challenge Participant Guidance.pdf  
  inflating: Dataset/Building_Footprint.kml  
  inflating: Dataset/Landsat_LST.ipynb  
  inflating: Dataset/NY_Mesonet_Weather.xlsx  
  inflating: Dataset/Sentinel2_GeoTIFF.ipynb  
  inflatin

In [ ]:
!cp "/content/drive/MyDrive/EY 2025/TrainTIFF.zip" /content/
!unzip TrainTIFF.zip
!rm TrainTIFF.zip
!mv Train TIFF

Archive:  TrainTIFF.zip
   creating: Train/
  inflating: Train/red_max.tiff      
  inflating: Train/green_std.tiff    
  inflating: Train/optional_vv_std.tiff  
  inflating: Train/qa_radsat_mean.tiff  
  inflating: Train/scl_std.tiff      
  inflating: Train/LST_Day_1km_std.tiff  
  inflating: Train/trad_min.tiff     
  inflating: Train/ndbi_std.tiff     
  inflating: Train/swir22_std.tiff   
  inflating: Train/LST_growth_min.tiff  
  inflating: Train/red_min.tiff      
  inflating: Train/Emis_31_max.tiff  
  inflating: Train/scl_median.tiff   
  inflating: Train/LST_Day_1km_min.tiff  
  inflating: Train/nir08_std.tiff    
  inflating: Train/nir08_max.tiff    
  inflating: Train/ndvi_median.tiff  
  inflating: Train/green_mean.tiff   
  inflating: Train/LST_growth_median.tiff  
  inflating: Train/optional_vh_mean.tiff  
  inflating: Train/ndvi_std.tiff     
  inflating: Train/wvp_std.tiff      
  inflating: Train/scl_mean.tiff     
  inflating: Train/atran_std.tiff    
  inflating: Tr

In [ ]:
import pandas as pd
import numpy as np
import rioxarray as rxr
from pyproj import Proj, Transformer
from tqdm import tqdm
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
tqdm.pandas()

tiff_files = list(Path("TIFF").rglob("*.tiff"))
tiff_files[:5]

[PosixPath('TIFF/atran_max.tiff'),
 PosixPath('TIFF/LST_Day_1km_var.tiff'),
 PosixPath('TIFF/ndbi_max.tiff'),
 PosixPath('TIFF/blue_std.tiff'),
 PosixPath('TIFF/Emis_32_median.tiff')]

In [ ]:
def create_csv(tiff_path, output_path, input_df):
    input_df = input_df.copy()
    path = Path(output_path)
    path.resolve().parent.mkdir(parents=True, exist_ok=True)
    mapping = map_satellite_data(tiff_path, input_df)
    mapping.to_csv(output_path, index=False)
    return


def create_dataset(df, tiff_files, typ=None, output_path=None):
    for file in tqdm(tiff_files, desc="Creating CSV features..."):
        # for file in tiff_files:
        create_csv(file, f"{typ}/{typ}_{file.stem}.csv", df)

    features = list(Path(typ).resolve().rglob(f"*{typ}_*.csv"))
    features = [pd.read_csv(f) for f in features if "Final" not in f.name]
    df_features = pd.concat([df, *features], axis=1)
    df_features.to_csv(output_path, index=False)
    return df_features


def map_point(row, data, radius_pixels):
    """
    Extract data within a specified radius (in pixels) around the nearest lat/long point.
    """
    latitudes = row["Latitude"]
    longitudes = row["Longitude"]

    # Find the nearest pixel coordinates
    nearest_point = data.sel(x=longitudes, y=latitudes, band=1, method="nearest")
    x_index = np.where(data.x == nearest_point.x)[0][0]
    y_index = np.where(data.y == nearest_point.y)[0][0]

    # Define the window around the nearest point
    x_slice = slice(
        max(0, x_index - radius_pixels), min(data.x.size, x_index + radius_pixels + 1)
    )
    y_slice = slice(
        max(0, y_index - radius_pixels), min(data.y.size, y_index + radius_pixels + 1)
    )

    # Extract the data within the window
    window_data = data.isel(x=x_slice, y=y_slice).values

    # Calculate the median of the extracted data (ignoring NaN or no-data values)
    median_value = np.nanmedian(window_data)
    return median_value


def map_satellite_data(tiff_path, input_df, radius_meters=50, meter_per_pixel=10):
    """
    Map satellite data to the input dataframe with a specified radius around each point.
    """
    # Open the TIFF file
    data = rxr.open_rasterio(tiff_path)
    tiff_crs = data.rio.crs

    # Calculate the radius in pixels
    radius_pixels = int(radius_meters / meter_per_pixel)

    # Initialize the coordinate transformer
    proj_wgs84 = Proj("epsg:4326")  # EPSG:4326 is the common lat/long CRS
    proj_tiff = Proj(tiff_crs)
    Transformer.from_proj(proj_wgs84, proj_tiff)

    # Apply the mapping function to each row in the dataframe
    values = input_df.apply(map_point, axis=1, data=data, radius_pixels=radius_pixels)

    # Create a new dataframe with the results
    df = pd.DataFrame(values, columns=[tiff_path.stem])
    return df

In [ ]:
!rm -r Train
!rm -r Submission


def prepare_train_sub():
    train_path = "/content/Training_data_uhi_index.csv"
    sub_path = "/content/Submission_template.csv"

    train_df = pd.read_csv(train_path)
    sub_df = pd.read_csv(sub_path)

    typ = "Train"
    out_path = f"{typ}/{typ}_Final.csv"
    Path(out_path).resolve().parent.mkdir(parents=True, exist_ok=True)
    print("Creating training data...")
    train_out = create_dataset(
        train_df, tiff_files=tiff_files, typ=typ, output_path=out_path
    )
    assert train_df.shape[0] == pd.read_csv(out_path).shape[0], (
        "Column not same between input and output after concatting the train_df!"
    )

    typ = "Submission"
    out_path = f"{typ}/{typ}_Final.csv"
    Path(out_path).resolve().parent.mkdir(parents=True, exist_ok=True)
    print("Creating submission data...")
    sub_out = create_dataset(
        sub_df, tiff_files=tiff_files, typ=typ, output_path=out_path
    )
    assert sub_df.shape[0] == pd.read_csv(out_path).shape[0], (
        "Column not same between input and output after concatting the sub_df!"
    )

    return train_out, sub_out


train_out, sub_out = prepare_train_sub()

Creating training data...


Creating CSV features...: 100%|██████████| 210/210 [56:04<00:00, 16.02s/it]


Creating submission data...


Creating CSV features...: 100%|██████████| 210/210 [11:48<00:00,  3.37s/it]


In [ ]:
!cp /content/Train/Train_Final.csv "/content/drive/MyDrive/EY 2025/Train_Final.csv" --remove-destination
!cp /content/Submission/Submission_Final.csv "/content/drive/MyDrive/EY 2025/Submission_Final.csv" --remove-destination

print("Done")

Done
